### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [6]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization
using Base.Threads
using LinearAlgebra
println("Running on $(Threads.nthreads()) threads")



Running on 10 threads


In [45]:
using SparseArrays
using LinearAlgebra
using Nemo

const F2 = GF(2)


function boundary_incidence_facets_to_ridges(facets::Vector{UInt16})
    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{UInt16, Int}()  # ridge -> row index
    ridges = Vector{UInt16}()
    for f in facets
        for i=0:((8*sizeof(f))-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                if !haskey(ridge_dict, r)
                    push!(ridges, r)
                    ridge_dict[r] = length(ridges)
                end
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(facets)
        for i=0:(8*sizeof(f)-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                i = ridge_dict[r]
                push!(I, i)
                push!(J, j)
                push!(V, true)
            end
        end
    end

    A = sparse(I, J, V, m, n)  # SparseMatrixCSC{Bool} 

    return ridges, A
end

function mod2_rank_nemo(A::SparseMatrixCSC)
    m, n = size(A)
    if m == 0 || n == 0
        return 0
    end

    M = zero_matrix(F2, m, n)

    # Fill matrix directly from sparse structure
    for col in 1:n
        for ptr in A.colptr[col]:(A.colptr[col+1]-1)
            row = A.rowval[ptr]
            M[row, col] = F2(1)
        end
    end

    return rank(M)
end


function is_mod2_sphere(top_facets::Vector{UInt16})

    isempty(top_facets) && return true

    d = count_ones(top_facets[1]) - 1

    current_faces = top_facets

    # ---- Step 1: Top homology β_d ----
    # β_d = (#d-faces - rank ∂_d)
    ridges, B = boundary_incidence_facets_to_ridges(current_faces)
    rank_prev = mod2_rank_nemo(B)

    n_d = length(current_faces)
    β_d = n_d - rank_prev
    β_d == 1 || return false

    current_faces = ridges

    # ---- Step 2: Middle dimensions ----
    # For i = d-1 down to 1:
    for dim = d-1:-1:1

        ridges, B = boundary_incidence_facets_to_ridges(current_faces)
        rank_curr = mod2_rank_nemo(B)

        n_i = length(current_faces)

        # β_i = (#i-faces - rank ∂_i) - rank ∂_{i+1}
        β_i = (n_i - rank_curr) - rank_prev
        β_i == 0 || return false

        rank_prev = rank_curr
        current_faces = ridges
    end

    # ---- Step 3: β_0 ----
    # β_0 = (#vertices) - rank ∂_1
    n_0 = length(current_faces)
    β_0 = n_0 - rank_prev
    β_0 == 1 || return false

    return true
end

function euler_characteristic_sphere(top_facets::Vector{UInt16})
    isempty(top_facets) && return 0

    d = count_ones(top_facets[1]) - 1

    # faces_by_dim[i] will store i-faces
    faces_by_dim = Vector{Set{UInt16}}(undef, d+1)

    # Top dimension
    faces_by_dim[d+1] = Set(top_facets)

    # Generate all lower-dimensional faces
    for dim = d:-1:1
        current_faces = faces_by_dim[dim+1]
        lower_faces = Set{UInt16}()

        for f in current_faces
            x = f
            while x != 0
                i = trailing_zeros(x)
                push!(lower_faces, f ⊻ (UInt16(1) << i))
                x &= x - 1
            end
        end

        faces_by_dim[dim] = lower_faces
    end

    # Compute χ = Σ (-1)^i f_i
    χ = 0
    for i in 0:d
        χ += (-1)^i * length(faces_by_dim[i+1])
    end

    return χ
end

function euler_sphere_test(top_facets::Vector{UInt16})
    isempty(top_facets) && return false
    d = count_ones(top_facets[1]) - 1
    χ = euler_characteristic(top_facets)
    return χ == 1 + (-1)^d
end


euler_sphere_test (generic function with 1 method)

In [22]:
function kernel_basis_mod2_sparse(A)
    m, n = size(A)

    # Store rows as sparse BitVectors (using sets of column indices)
    rows = [Set{Int}() for _ in 1:m]
    
    # Build rows from A mod 2
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                if j in rows[i]
                    delete!(rows[i], j)  # XOR: 1⊕1=0
                else
                    push!(rows[i], j)     # XOR: 0⊕1=1
                end
            end
        end
    end

    # RREF over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    
    @inbounds for col in 1:n
        row > m && break

        # Find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if col in rows[r]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # Swap rows
        if piv != row
            rows[row], rows[piv] = rows[piv], rows[row]
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # Eliminate this column in all other rows (RREF)
        pivot_row = rows[row]
        for r in 1:m
            if r != row && col in rows[r]
                # XOR rows[r] with pivot_row
                symdiff!(rows[r], pivot_row)
            end
        end

        row += 1
    end

    # Free columns
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x_set = Set{Int}([f])  # Sparse representation of x

        # Back-substitute using RREF rows
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            
            # Compute parity: count elements in rows[r] ∩ x_set, excluding c
            row_r = rows[r]
            parity = false
            for j in row_r
                if j != c && j in x_set
                    parity = !parity
                end
            end
            
            # Set x[c] based on parity
            if parity
                push!(x_set, c)
            else
                delete!(x_set, c)
            end
        end

        # Convert sparse set to BitVector
        x = falses(n)
        for j in x_set
            x[j] = true
        end
        push!(basis, x)
    end

    return basis
end

kernel_basis_mod2_sparse (generic function with 1 method)

In [23]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    # println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [16]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    # print(b)
    d = dim(K)
    for i in 1:d
        if i == d
            b[i] == 1 || return false
        else
            b[i] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [ ]:


# function kernel_basis_echelon_prioritize(B, S)
#     """
#     Convert kernel basis B to echelon form over GF(2), prioritizing columns in S.
#     Returns (B_ech, pivots) where:
#     - B_ech is the echelon basis
#     - pivots[i] is the pivot position (leading 1) of B_ech[i]
#     - Columns in S are processed first to maximize forced coefficients
#     """
#     isempty(B) && return (BitVector[], Int[])
    
#     n = length(B[1])
#     k = length(B)
    
#     # Copy basis vectors
#     B_ech = [copy(b) for b in B]
#     pivots = Int[]
    
#     # Build column order: S columns first, then others
#     cols_in_S = [j for j in 1:n if S[j]]
#     cols_not_in_S = [j for j in 1:n if !S[j]]
#     col_order = vcat(cols_in_S, cols_not_in_S)
    
#     current_row = 1
#     for col in col_order
#         current_row > k && break
        
#         # Find a vector with 1 at position col (at or below current_row)
#         piv = 0
#         for r in current_row:k
#             if B_ech[r][col]
#                 piv = r
#                 break
#             end
#         end
        
#         piv == 0 && continue  # No pivot in this column
        
#         # Swap to current position
#         if piv != current_row
#             B_ech[current_row], B_ech[piv] = B_ech[piv], B_ech[current_row]
#         end
        
#         push!(pivots, col)
        
#         # Eliminate this column in all other rows
#         for r in 1:k
#             if r != current_row && B_ech[r][col]
#                 B_ech[r] .⊻= B_ech[current_row]
#             end
#         end
        
#         current_row += 1
#     end
    
#     return (B_ech, pivots)
# end


# function enumerate_kernel_with_constraints(A, B, S)
#     m, n = size(A)
#     results = BitVector[]
    
#     if isempty(B)
#         x = falses(n)
#         if all(.!S) && check_Ax_is_02(A, x)
#             push!(results, x)
#         end
#         return results
#     end
    
#     # Convert basis to echelon form, prioritizing S columns
#     B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
#     k = length(B_ech)
    
#     # Determine forced and free coefficients
#     forced_lambda = falses(k)
#     free_indices = Int[]
    
#     for i in 1:k
#         piv = pivots[i]
#         if S[piv]
#             forced_lambda[i] = true
#         else
#             push!(free_indices, i)
#         end
#     end
    
#     # Compute base vector from forced coefficients
#     x_forced = falses(n)
#     for i in 1:k
#         if forced_lambda[i]
#             x_forced .⊻= B_ech[i]
#         end
#     end
    
#     # Early exit if x_forced doesn't satisfy S
#     for j in 1:n
#         if S[j] && !x_forced[j]
#             return results
#         end
#     end
    
#     num_free = length(free_indices)
    
#     # Early exit if no free variables
#     if num_free == 0
#         if check_Ax_is_02(A, x_forced)
#             push!(results, copy(x_forced))
#         end
#         return results
#     end
    
#     # Pre-allocate with size hint
#     sizehint!(results, min(2^num_free, 1000))
    
#     # Gray code enumeration
#     x = copy(x_forced)
    
#     # First iteration (all free vars = 0)
#     if check_Ax_is_02(A, x)
#         push!(results, copy(x))
#     end
    
#     # Enumerate using Gray code
#     for i in 1:(2^num_free - 1)
#         gray = i ⊻ (i >> 1)
#         gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
#         changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
#         idx = free_indices[changed_bit]
#         x .⊻= B_ech[idx]
        
#         if check_Ax_is_02(A, x)
#             push!(results, copy(x))
#         end
#     end
    
#     return results
# end

# function check_Ax_is_02(A, x)
#     m, n = size(A)
#     result = zeros(Int, m)
    
#     for j in 1:n
#         if x[j]
#             for p in nzrange(A, j)
#                 i = rowvals(A)[p]
#                 result[i] += A.nzval[p]
                
#                 if result[i] > 2
#                     return false
#                 end
#             end
#         end
#     end
#     return true

# end

check_Ax_is_02 (generic function with 2 methods)

In [43]:
function enumerate_kernel_with_constraints(A, B, S)
    m, n = size(A)
    results = BitVector[]
    
    if isempty(B)
        x = falses(n)
        if all(.!S) && check_Ax_is_02(A, x)
            push!(results, x)
        end
        return results
    end
    
    # Compute fixpoint of constraints
    S_fixed, T_fixed = compute_constraint_fixpoint(A, B, S)
    
    # Convert basis to echelon form with all constraints
    B_ech, pivots = kernel_basis_echelon_prioritize_with_constraints(B, S_fixed, T_fixed)
    k = length(B_ech)
    
    # Determine forced and free coefficients
    forced_one = falses(k)   # Must be 1
    forced_zero = falses(k)  # Must be 0
    free_indices = Int[]
    
    for i in 1:k
        piv = pivots[i]
        if S_fixed[piv]
            forced_one[i] = true
        elseif T_fixed[piv]
            forced_zero[i] = true
        else
            push!(free_indices, i)
        end
    end
    
    # Compute base vector from forced_one coefficients
    x_forced = falses(n)
    for i in 1:k
        if forced_one[i]
            x_forced .⊻= B_ech[i]
        end
    end
    
    # Verify constraints are satisfied
    for j in 1:n
        if S_fixed[j] && !x_forced[j]
            return results  # Inconsistent
        end
        if T_fixed[j] && x_forced[j]
            return results  # Inconsistent
        end
    end
    
    num_free = length(free_indices)
    
    # Early exit if no free variables
    if num_free == 0
        if check_Ax_is_02(A, x_forced)
            push!(results, copy(x_forced))
        end
        return results
    end
    
    # Pre-allocate with size hint
    sizehint!(results, min(2^num_free, 1000))
        
    # Gray code enumeration
    x = copy(x_forced)
    
    # First iteration
    if check_Ax_is_02(A, x)
        push!(results, copy(x))
    end
    
    # Enumerate using Gray code
    for i in 1:(2^num_free - 1)
        gray = i ⊻ (i >> 1)
        gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
        changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
        idx = free_indices[changed_bit]
        x .⊻= B_ech[idx]
        
        if check_Ax_is_02(A, x)
            push!(results, copy(x))
        end
    end
    
    return results
end

"""
Compute the fixpoint of constraints by alternating:
- S (must be 1) → find saturated rows → T (must be 0)
- T (must be 0) → find rows at 1 → U (must be 1 to reach 2)
- U → new T, etc.
"""
function compute_constraint_fixpoint(A, B, S_init)
    m, n = size(A)
    
    S = copy(S_init)
    T = falses(n)
    
    max_iterations = 100  # Prevent infinite loops
    
    for iter in 1:max_iterations
        S_old = copy(S)
        T_old = copy(T)
        
        # Compute what x would be if we only set S positions
        x_partial = compute_partial_solution(B, S, T)
        
        if isnothing(x_partial)
            # Constraints are inconsistent
            return (S, T)
        end
        
        # Compute A * x_partial
        Ax = zeros(Int, m)
        for j in 1:n
            if x_partial[j]
                for p in nzrange(A, j)
                    i = rowvals(A)[p]
                    Ax[i] += A.nzval[p]
                end
            end
        end
        
        # Find new constraints
        # Rule 1: Saturated rows (Ax[i] == 2) force remaining nonzero columns to 0
        for i in 1:m
            if Ax[i] == 2
                for p in nzrange(A, i)  # Need CSR format - convert
                    # For column j where A[i,j] != 0
                end
            end
        end
        
        # Better: iterate over columns
        for j in 1:n
            if S[j] || T[j]
                continue  # Already constrained
            end
            
            # Check if j must be 0 (appears in saturated row)
            must_be_zero = false
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                if Ax[i] == 2 && A.nzval[p] != 0
                    must_be_zero = true
                    break
                end
            end
            
            if must_be_zero
                T[j] = true
                continue
            end
            
            # Check if j must be 1 (needed to saturate a row at 1)
            must_be_one = false
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                if Ax[i] == 1 && A.nzval[p] != 0
                    # Check if j is the ONLY way to saturate row i
                    # Count how many unset columns could saturate row i
                    candidates = 0
                    for p2 in nzrange(A, i)  # Wrong - need CSR
                        # This is CSC, need to find all j' where A[i,j'] != 0
                    end
                    # For now, mark as must be 1 if it's in a row with value 1
                    must_be_one = true
                    break
                end
            end
            
            if must_be_one
                S[j] = true
            end
        end
        
        # Check convergence
        if S == S_old && T == T_old
            break
        end
    end
    
    return (S, T)
end

"""
Compute a partial solution satisfying S (must be 1) and T (must be 0) constraints.
Returns nothing if constraints are inconsistent with kernel.
"""
function compute_partial_solution(B, S, T)
    isempty(B) && return falses(length(S))
    
    n = length(B[1])
    
    # Use echelon form to find a solution
    B_ech, pivots = kernel_basis_echelon_prioritize_with_constraints(B, S, T)
    
    x = falses(n)
    
    # Set coefficients based on constraints
    for i in 1:length(pivots)
        piv = pivots[i]
        if S[piv]
            x .⊻= B_ech[i]
        elseif T[piv]
            # Coefficient is 0, don't add
        else
            # Free - leave at 0 for partial solution
        end
    end
    
    # Check if S constraints are satisfied
    for j in 1:n
        if S[j] && !x[j]
            return nothing
        end
        if T[j] && x[j]
            return nothing
        end
    end
    
    return x
end

"""
Echelon form prioritizing: S columns (must be 1), then T columns (must be 0), then others.
"""
function kernel_basis_echelon_prioritize_with_constraints(B, S, T)
    isempty(B) && return (BitVector[], Int[])
    
    n = length(B[1])
    k = length(B)
    
    # Copy basis vectors
    B_ech = [copy(b) for b in B]
    pivots = Int[]
    
    # Build column order: S first, then T, then others
    cols_in_S = [j for j in 1:n if S[j]]
    cols_in_T = [j for j in 1:n if !S[j] && T[j]]
    cols_other = [j for j in 1:n if !S[j] && !T[j]]
    col_order = vcat(cols_in_S, cols_in_T, cols_other)
    
    current_row = 1
    for col in col_order
        current_row > k && break
        
        # Find a vector with 1 at position col
        piv = 0
        for r in current_row:k
            if B_ech[r][col]
                piv = r
                break
            end
        end
        
        piv == 0 && continue
        
        # Swap
        if piv != current_row
            B_ech[current_row], B_ech[piv] = B_ech[piv], B_ech[current_row]
        end
        
        push!(pivots, col)
        
        # Eliminate
        for r in 1:k
            if r != current_row && B_ech[r][col]
                B_ech[r] .⊻= B_ech[current_row]
            end
        end
        
        current_row += 1
    end
    
    return (B_ech, pivots)
end

function check_Ax_is_02(A, x)
    m, n = size(A)
    result = zeros(Int, m)
    
    for j in 1:n
        if x[j]
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                result[i] += A.nzval[p]
                
                if result[i] > 2
                    return false
                end
            end
        end
    end
    return true
end

"""
Build CSR (row-sparse) representation of A for efficient row iteration.
Returns (row_ptrs, col_indices, values) where row i has nonzeros in col_indices[row_ptrs[i]:row_ptrs[i+1]-1]
"""
function build_csr(A)
    m, n = size(A)
    
    # Count nonzeros per row
    row_counts = zeros(Int, m)
    for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            row_counts[i] += 1
        end
    end
    
    # Build row pointers
    row_ptrs = zeros(Int, m + 1)
    row_ptrs[1] = 1
    for i in 1:m
        row_ptrs[i+1] = row_ptrs[i] + row_counts[i]
    end
    
    # Fill col_indices and values
    nnz_total = row_ptrs[end] - 1
    col_indices = zeros(Int, nnz_total)
    values = zeros(Int, nnz_total)
    current_pos = copy(row_ptrs[1:m])
    
    for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            pos = current_pos[i]
            col_indices[pos] = j
            values[pos] = A.nzval[p]
            current_pos[i] += 1
        end
    end
    
    return (row_ptrs, col_indices, values)
end

build_csr

In [ ]:
# function enumerate_kernel_with_constraints(A, B, S)
#     m, n = size(A)
#     results = BitVector[]
    
#     if isempty(B)
#         x = falses(n)
#         if all(.!S) && check_Ax_is_02(A, x)
#             push!(results, x)
#         end
#         return results
#     end
    
#     # Convert basis to echelon form (using BitVectors)
#     B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
#     k = length(B_ech)
    
#     # Determine forced and free coefficients
#     forced_lambda = falses(k)
#     free_indices = Int[]
    
#     for i in 1:k
#         piv = pivots[i]
#         if S[piv]
#             forced_lambda[i] = true
#         else
#             push!(free_indices, i)
#         end
#     end
    
#     # Compute base vector from forced coefficients (stays as BitVector)
#     x_forced = falses(n)
#     for i in 1:k
#         if forced_lambda[i]
#             x_forced .⊻= B_ech[i]
#         end
#     end
    
#     # Early exit if x_forced doesn't satisfy S
#     for j in 1:n
#         if S[j] && !x_forced[j]
#             return results
#         end
#     end
    
#     num_free = length(free_indices)
    
#     # Early exit if no free variables
#     if num_free == 0
#         if check_Ax_is_02(A, x_forced)
#             push!(results, copy(x_forced))
#         end
#         return results
#     end
    
#     # Choose integer type based on num_free (not k or n!)
#     if num_free <= 64
#         return enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, UInt64)
#     elseif num_free <= 128
#         return enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, UInt128)
#     else
#         # Fall back to BitVector for large num_free
#         return enumerate_with_bitvector(A, B_ech, x_forced, free_indices, num_free)
#     end
# end

# function enumerate_with_int_encoding(A, B_ech, x_forced, free_indices, num_free, ::Type{IntType}) where IntType <: Unsigned
#     n = length(x_forced)
#     results = BitVector[]
#     sizehint!(results, min(2^num_free, 1000))
    
#     # Convert free basis vectors and x_forced to integers
#     B_free_int = IntType[bitvector_to_uint(B_ech[idx], IntType) for idx in free_indices]
#     x_forced_int = bitvector_to_uint(x_forced, IntType)
    
#     x_int = x_forced_int
    
#     # First iteration
#     if check_Ax_is_02_int(A, x_int, n)
#         push!(results, uint_to_bitvector(x_int, n))
#     end
    
#     # Gray code enumeration
#     for i in 1:(2^num_free - 1)
#         gray = i ⊻ (i >> 1)
#         gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
#         changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
#         # Fast integer XOR
#         x_int ⊻= B_free_int[changed_bit]
        
#         if check_Ax_is_02_int(A, x_int, n)
#             push!(results, uint_to_bitvector(x_int, n))
#         end
#     end
    
#     return results
# end

# function check_Ax_is_02_int(A, x_int::T, n::Int) where T <: Unsigned
#     m = size(A, 1)
#     result = zeros(Int, m)
    
#     # Iterate over set bits
#     x_temp = x_int
#     bit_pos = 0
    
#     while x_temp != 0
#         tz = trailing_zeros(x_temp)
#         bit_pos += tz
#         j = bit_pos + 1
        
#         if j <= n
#             for p in nzrange(A, j)
#                 i = rowvals(A)[p]
#                 result[i] += A.nzval[p]
                
#                 if result[i] > 2
#                     return false
#                 end
#             end
#         end
        
#         x_temp >>= (tz + 1)
#         bit_pos += 1
#     end
    
#     return true
# end

# function enumerate_with_bitvector(A, B_ech, x_forced, free_indices, num_free)
#     results = BitVector[]
#     sizehint!(results, min(2^num_free, 1000))
    
#     x = copy(x_forced)
    
#     if check_Ax_is_02(A, x)
#         push!(results, copy(x))
#     end
    
#     for i in 1:(2^num_free - 1)
#         gray = i ⊻ (i >> 1)
#         gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
#         changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
#         idx = free_indices[changed_bit]
#         x .⊻= B_ech[idx]
        
#         if check_Ax_is_02(A, x)
#             push!(results, copy(x))
#         end
#     end
    
#     return results
# end

# function check_Ax_is_02(A, x::BitVector)
#     m, n = size(A)
#     result = zeros(Int, m)
    
#     for j in 1:n
#         if x[j]
#             for p in nzrange(A, j)
#                 i = rowvals(A)[p]
#                 result[i] += A.nzval[p]
                
#                 if result[i] > 2
#                     return false
#                 end
#             end
#         end
#     end
#     return true
# end

# function bitvector_to_uint(bv::BitVector, ::Type{T}) where T <: Unsigned
#     n = length(bv)
#     result = T(0)
#     for i in 1:min(n, 8 * sizeof(T))
#         if bv[i]
#             result |= T(1) << (i - 1)
#         end
#     end
#     return result
# end

# function uint_to_bitvector(x::T, n::Int) where T <: Unsigned
#     bv = falses(n)
#     for i in 1:min(n, 8 * sizeof(T))
#         if (x >> (i - 1)) & 1 == 1
#             bv[i] = true
#         end
#     end
#     return bv
# end

uint_to_bitvector (generic function with 1 method)

In [ ]:
using Profile
using ProgressMeter
using Polymake



global mat_DB_bin = open("rank_4_simple_bin_mat_DB_bin.jls", "r") do io
    deserialize(io)
end

global iso_DB = open("rank_4_iso_DB_m_7-15.jls", "r") do io
    deserialize(io)
end

function subset_bitvector(superset::Vector{UInt16}, subset::Vector{UInt16})
    n = length(superset)
    result = falses(n)
    
    j = 1  # index dans subset
    for i in 1:n
        if j <= length(subset) && superset[i] == subset[j]
            result[i] = true
            j += 1
        end
    end
    
    return result
end


function build_finalDB_single_v!(pseudo_manifolds_DB::Dict{Int,Vector{Set{BitVector}}},mat_DB::Dict{Int,Vector{Vector{UInt16}}},iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}},mmax)
    mmin = minimum(collect(keys(mat_DB)))
    for m=mmin:mmax
        println(m)
        pseudo_manifolds_DB[m] = Vector{Set{BitVector}}()
        for (l,bases) in enumerate(mat_DB[m])
            # display(bases)
            V = reduce(|,bases)
            compl_bases = [base⊻V for base in bases] # we need to complement to get the correct boundary matrix
            ridges, A = boundary_incidence_facets_to_ridges(compl_bases)  
            basis = kernel_basis_mod2_sparse(A)
            push!(pseudo_manifolds_DB[m], Set{BitVector}())
            if m==mmin
                mandatory_facets_bit=falses(length(bases))
                all_solutions_bit = kernel_elements_with_Ax_in_0_or_2(A,basis)
                for K_bit in all_solutions_bit
                    facets_bin = [facet_bin for facet_bin in compl_bases[findall(K_bit)]]
                    if !euler_sphere_test(facets_bin)
                        continue
                    end
                    if is_mod2_sphere(facets_bin)
                        push!(pseudo_manifolds_DB[m][l],K_bit)
                    end
                end
            else
                index_contraction, v_contract, perm = iso_DB[m][l]
                @showprogress desc="Number of links $(length(pseudo_manifolds_DB[m-1][index_contraction]))" for L in pseudo_manifolds_DB[m-1][index_contraction]
                    mandatory_facets = [reduce(|,[UInt16(1)<<(perm[i]-1) for i=1:(sizeof(facet_L)) if (facet_L>>(i-1)&1)==1],init=UInt16(0))⊻(UInt16(1)<<(v_contract-1)) for facet_L in mat_DB[m-1][index_contraction][findall(L)]]
                    # print(mandatory_facets)
                    mandatory_facets_bit = subset_bitvector(bases, mandatory_facets)
                    t1 = time()
                    all_solutions_bit = enumerate_kernel_with_constraints(A,basis,mandatory_facets_bit)
                    if (time() - t1)> 1
                        println("Enum: ", time() - t1, " seconds")
                    end
                    if m<9
                        for K_bit in all_solutions_bit
                            facets_bin = [facet_bin for facet_bin in compl_bases[findall(K_bit)]]
                            if !euler_sphere_test(facets_bin)
                                continue
                            end
                            if is_mod2_sphere(facets_bin)
                                push!(pseudo_manifolds_DB[m][l],K_bit)
                            end
                        end
                    else
                        union!(pseudo_manifolds_DB[m][l],all_solutions_bit)
                    end
                end
            end
        end
    end
end

global pseudo_manifolds_DB = Dict{Int,Vector{Set{BitVector}}}()

build_finalDB_single_v!(pseudo_manifolds_DB,mat_DB_bin,iso_DB,15)

open("Pic_4_DB_7-15.jls", "w") do io
    serialize(io, pseudo_manifolds_DB)
end




6
7


Number of links 36 100%|█████████████████████████████████| Time: 0:00:00


8


Number of links 39 100%|█████████████████████████████████| Time: 0:00:00
Number of links 127 100%|████████████████████████████████| Time: 0:00:17
Number of links 232 100%|████████████████████████████████| Time: 0:06:26
Number of links 39 100%|█████████████████████████████████| Time: 0:00:03
Number of links 115 100%|████████████████████████████████| Time: 0:00:39
Number of links 115 100%|████████████████████████████████| Time: 0:00:20


9


Number of links 5148 100%|███████████████████████████████| Time: 0:03:12
Number of links 1908   4%|█▏                             |  ETA: 0:19:11